In [ ]:
%%writefile matmul.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// CUDA Kernel
__global__ void matmul(int* A, int* B, int* C, int N) {

    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if (Row < N && Col < N) {

        int sum = 0;

        // Multiply row of A with column of B
        for (int k = 0; k < N; k++) {
            sum += A[Row * N + k] * B[k * N + Col];
        }

        C[Row * N + Col] = sum;
    }
}

int main() {

    int N = 4;
    int size = N * N * sizeof(int);

    int *A, *B, *C;
    int *dev_A, *dev_B, *dev_C;

    // Allocate memory on Host
    cudaMallocHost(&A, size);
    cudaMallocHost(&B, size);
    cudaMallocHost(&C, size);

    // Initialize Matrix A
    cout << "Matrix A:\n";

    for (int i = 0; i < N; i++) {

        for (int j = 0; j < N; j++) {

            A[i * N + j] = i + j;

            cout << A[i * N + j] << " ";
        }

        cout << endl;
    }

    // Initialize Matrix B
    cout << "\nMatrix B:\n";

    for (int i = 0; i < N; i++) {

        for (int j = 0; j < N; j++) {

            B[i * N + j] = i * j;

            cout << B[i * N + j] << " ";
        }

        cout << endl;
    }

    // Allocate memory on Device
    cudaMalloc(&dev_A, size);
    cudaMalloc(&dev_B, size);
    cudaMalloc(&dev_C, size);

    // Copy Host -> Device
    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    // Define block and grid size
    dim3 blockSize(16, 16);

    dim3 gridSize(
        (N + 15) / 16,
        (N + 15) / 16
    );

    // Launch kernel
    matmul<<<gridSize, blockSize>>>(
        dev_A,
        dev_B,
        dev_C,
        N
    );

    // Copy Device -> Host
    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    // Print result matrix
    cout << "\nResult Matrix (C = A x B):\n";

    for (int i = 0; i < N; i++) {

        for (int j = 0; j < N; j++) {

            cout << C[i * N + j] << " ";
        }

        cout << endl;
    }

    // Free device memory
    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    // Free host memory
    cudaFreeHost(A);
    cudaFreeHost(B);
    cudaFreeHost(C);

    return 0;
}

Writing matmul.cu


In [ ]:
!nvcc matmul.cu -o matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matmul

/bin/bash: line 1: ./matmul: No such file or directory
